# Center of Mass + Velocity menggunakan Machine Learning

**Algoritma:** Linear Regression  
**Jenis pembelajaran:** Supervised Learning (regression)  
**Dataset:** Synthetic Dataset yang dibuat otomatis dari simulasi fisika.

Notebook ini menambahkan model untuk memprediksi posisi pusat massa pada waktu mendatang. Dataset dibuat sendiri, jadi tidak memerlukan unduhan dataset dari internet.

> Catatan akademik: simulasi menggunakan gerak berkecepatan konstan. Model ML adalah demonstrasi supervised learning pada data sintetis. Karena persamaan fisikanya sudah diketahui, hasil ML dibandingkan dengan hasil rumus fisika.


## 1. Persiapan library
Jalankan cell ini jika library belum tersedia.

In [ ]:
%pip -q install numpy pandas matplotlib scikit-learn


## 2. Import library dan fungsi fisika

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def hitung_pusat_massa(posisi, massa):
    posisi = np.asarray(posisi, dtype=float)
    massa = np.asarray(massa, dtype=float)
    if np.any(massa <= 0):
        raise ValueError("Massa harus lebih besar dari nol.")
    return np.sum(posisi * massa[:, None], axis=0) / np.sum(massa)

def hitung_kecepatan_pusat_massa(kecepatan, massa):
    kecepatan = np.asarray(kecepatan, dtype=float)
    massa = np.asarray(massa, dtype=float)
    if np.any(massa <= 0):
        raise ValueError("Massa harus lebih besar dari nol.")
    return np.sum(kecepatan * massa[:, None], axis=0) / np.sum(massa)

FEATURE_COLUMNS = [
    "cm_x_awal_terukur", "cm_y_awal_terukur", "cm_z_awal_terukur",
    "perpindahan_x_terukur", "perpindahan_y_terukur", "perpindahan_z_terukur"
]
TARGET_COLUMNS = ["cm_x_masa_depan", "cm_y_masa_depan", "cm_z_masa_depan"]


## 3. Membuat dataset sintetis

Setiap baris mewakili satu skenario dengan 2–6 objek. Massa, posisi, kecepatan, dan waktu dibuat secara acak. Target posisi pusat massa masa depan dihitung dari simulasi fisika. Gangguan kecil ditambahkan pada fitur input untuk meniru ketidakpastian pengukuran.


In [ ]:
def buat_dataset_sintetis(jumlah_sampel=3000, noise_std=0.05, random_state=42):
    rng = np.random.default_rng(random_state)
    baris = []

    for _ in range(jumlah_sampel):
        jumlah_objek = int(rng.integers(2, 7))
        massa = rng.uniform(0.5, 10.0, size=jumlah_objek)
        posisi = rng.uniform(-20.0, 20.0, size=(jumlah_objek, 3))
        kecepatan = rng.uniform(-5.0, 5.0, size=(jumlah_objek, 3))
        waktu = float(rng.uniform(0.1, 5.0))

        cm_awal = hitung_pusat_massa(posisi, massa)
        v_cm = hitung_kecepatan_pusat_massa(kecepatan, massa)
        perpindahan = v_cm * waktu
        cm_masa_depan = cm_awal + perpindahan

        cm_terukur = cm_awal + rng.normal(0, noise_std, size=3)
        perpindahan_terukur = perpindahan + rng.normal(0, noise_std, size=3)

        baris.append({
            "jumlah_objek": jumlah_objek,
            "waktu_s": waktu,
            "kecepatan_cm_x": v_cm[0], "kecepatan_cm_y": v_cm[1], "kecepatan_cm_z": v_cm[2],
            "cm_x_awal_terukur": cm_terukur[0],
            "cm_y_awal_terukur": cm_terukur[1],
            "cm_z_awal_terukur": cm_terukur[2],
            "perpindahan_x_terukur": perpindahan_terukur[0],
            "perpindahan_y_terukur": perpindahan_terukur[1],
            "perpindahan_z_terukur": perpindahan_terukur[2],
            "cm_x_masa_depan": cm_masa_depan[0],
            "cm_y_masa_depan": cm_masa_depan[1],
            "cm_z_masa_depan": cm_masa_depan[2]
        })
    return pd.DataFrame(baris)

dataset = buat_dataset_sintetis(jumlah_sampel=3000)
dataset.to_csv("dataset_center_of_mass_ml.csv", index=False)
print("Ukuran dataset:", dataset.shape)
display(dataset.head())


## 4. Training model Linear Regression

Data dibagi menjadi 80% training dan 20% testing. Fitur yang dipakai adalah posisi pusat massa awal dan perpindahan terukur. Targetnya adalah posisi pusat massa masa depan pada sumbu X, Y, dan Z.


In [ ]:
X = dataset[FEATURE_COLUMNS]
y = dataset[TARGET_COLUMNS]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)
y_prediksi = model.predict(X_test)

print("Jumlah data training:", len(X_train))
print("Jumlah data testing :", len(X_test))
print("Training selesai.")


## 5. Evaluasi model

MAE dan RMSE mengukur besar kesalahan prediksi. R² mendekati 1 menunjukkan kecocokan tinggi pada data uji.

In [ ]:
mae = mean_absolute_error(y_test, y_prediksi)
rmse = np.sqrt(mean_squared_error(y_test, y_prediksi))
r2 = r2_score(y_test, y_prediksi)

print(f"MAE       : {mae:.6f}")
print(f"RMSE      : {rmse:.6f}")
print(f"R-squared : {r2:.6f}")

aktual = y_test.to_numpy()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, ax in enumerate(axes):
    ax.scatter(aktual[:, i], y_prediksi[:, i], alpha=0.5)
    low = min(aktual[:, i].min(), y_prediksi[:, i].min())
    high = max(aktual[:, i].max(), y_prediksi[:, i].max())
    ax.plot([low, high], [low, high], "r--")
    ax.set_xlabel(f"Aktual {['X','Y','Z'][i]}")
    ax.set_ylabel(f"Prediksi {['X','Y','Z'][i]}")
    ax.set_title(f"Sumbu {['X','Y','Z'][i]}")
    ax.grid(True)
plt.suptitle("Nilai aktual vs prediksi Linear Regression")
plt.tight_layout()
plt.show()


## 6. Mencoba prediksi pada skenario baru

Ubah nilai massa, posisi, kecepatan, dan waktu untuk mencoba skenario lain. Hasil ML dibandingkan dengan hasil rumus fisika gerak berkecepatan konstan.

\[\vec r_{CM}(t)=\vec r_{CM}(0)+\vec v_{CM}t\]


In [ ]:
# Contoh 3 objek. Ubah nilai-nilai ini untuk skenario Anda sendiri.
massa_baru = np.array([2.0, 3.0, 5.0])
posisi_baru = np.array([[1.0, 2.0, 0.0], [4.0, 1.0, 2.0], [2.0, 5.0, 3.0]])
kecepatan_baru = np.array([[1.0, 0.5, 0.0], [0.0, 1.0, 0.5], [0.5, 0.0, 1.0]])
waktu_baru = 2.0

cm_awal_baru = hitung_pusat_massa(posisi_baru, massa_baru)
v_cm_baru = hitung_kecepatan_pusat_massa(kecepatan_baru, massa_baru)
perpindahan_baru = v_cm_baru * waktu_baru
fitur_baru = pd.DataFrame([[*cm_awal_baru, *perpindahan_baru]], columns=FEATURE_COLUMNS)

prediksi_ml = model.predict(fitur_baru)[0]
prediksi_fisika = cm_awal_baru + perpindahan_baru

print("Pusat massa awal         :", np.round(cm_awal_baru, 4))
print("Kecepatan pusat massa    :", np.round(v_cm_baru, 4))
print("Hasil rumus fisika       :", np.round(prediksi_fisika, 4))
print("Prediksi Machine Learning:", np.round(prediksi_ml, 4))
print("Selisih absolut          :", np.round(np.abs(prediksi_ml - prediksi_fisika), 4))

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(*cm_awal_baru, color="blue", s=90, label="Pusat massa awal")
ax.scatter(*prediksi_fisika, color="green", marker="^", s=110, label="Rumus fisika")
ax.scatter(*prediksi_ml, color="red", marker="x", s=110, label="Prediksi ML")
ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
ax.set_title("Perbandingan posisi pusat massa")
ax.legend()
plt.show()


## 7. Kesimpulan awal

1. Dataset dibuat otomatis dari simulasi sehingga tidak memerlukan dataset eksternal.
2. Linear Regression mempelajari pemetaan dari posisi awal dan perpindahan ke posisi pusat massa masa depan.
3. MAE, RMSE, dan R² digunakan untuk mengevaluasi hasil pada data testing.
4. Karena data sintetis mengikuti persamaan fisika gerak berkecepatan konstan, prediksi ML seharusnya mendekati hasil rumus fisika.
5. Keterbatasan: ini merupakan demonstrasi supervised learning. Untuk penelitian yang lebih kuat, model perlu diuji pada data eksperimen/sensor nyata atau dinamika yang lebih kompleks.
